# Look Like Me — VGGFace2 ResNet50 Fine-Tuning

Fine-tunes VGGFace2 ResNet50 on pseudo-labeled CelebA triplets using Triplet Margin Loss.

**Configuration**
- Dataset: 5,000 CelebA images, filtered with MTCNN (single face, confidence ≥ 0.90)
- Positive pair: highest-similarity match (> 0.60) from a different identity
- Negative pair: random image with similarity < 0.30
- Trainable layers: `layer4` only (rest of the backbone frozen)
- Loss: Triplet Margin Loss, margin = 0.3
- Optimizer: Adam, lr = 1e-5
- Epochs: 5, Batch size: 32

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive')

!pip install -q Pillow==10.2.0
!pip install -q facenet-pytorch

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
import os, cv2, pickle, random, json
import numpy as np
import torch
import torch.nn as nn
import torchvision.models as models
from torch.utils.data import Dataset, DataLoader
from PIL import Image
from facenet_pytorch import MTCNN as FaceMTCNN

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

[transformers] Disabling PyTorch because PyTorch >= 2.4 is required but found 2.2.2
[transformers] PyTorch was not found. Models won't be available and only tokenizers, configuration and file/data utilities can be used.


Device: cuda


## 2. Extract CelebA Images

In [3]:
import zipfile

zip_path     = '/content/drive/MyDrive/CelebA/img_align_celeba.zip'
extract_path = '/content/celeba_images'
IMG_DIR      = f'{extract_path}/img_align_celeba'

os.makedirs(extract_path, exist_ok=True)
with zipfile.ZipFile(zip_path, 'r') as z:
    z.extractall(extract_path)
print(f"Total images available: {len(os.listdir(IMG_DIR))}")

Total images available: 202599


## 3. Load the Pretrained VGGFace2 ResNet50

In [4]:
WEIGHTS_PATH = '/content/drive/MyDrive/CelebA/resnet50_ft_weight.pkl'

# Caffe-format pretrained weights (MS1M -> VGGFace2), loaded via pickle
with open(WEIGHTS_PATH, 'rb') as f:
    weights = pickle.load(f, encoding='latin1')
weights_tensor = {k: torch.from_numpy(v) for k, v in weights.items()}

model = models.resnet50(weights=None)
model.fc = nn.Linear(2048, 8631)
model.load_state_dict(weights_tensor, strict=False)
model.fc = nn.Identity()
model.eval().to(device)
print("VGGFace2 ResNet50 loaded.")

VGGFace2 ResNet50 loaded.


## 4. Select 5,000 Images, Filtered with MTCNN
Samples 7,000 candidates and keeps the first 5,000 that pass MTCNN validation (single face, confidence ≥ 0.90).

In [5]:
mtcnn = FaceMTCNN(keep_all=True, device=device, min_face_size=40)

CONFIDENCE_THRESHOLD = 0.90
TARGET               = 5000
SAMPLE_SIZE          = 7000

random.seed(42)
all_images = sorted(os.listdir(IMG_DIR))
sample_7k  = random.sample(all_images, SAMPLE_SIZE)

filtered = []
rejected = 0

for i, fname in enumerate(sample_7k):
    try:
        img_pil = Image.open(f'{IMG_DIR}/{fname}').convert('RGB')
        boxes, probs = mtcnn.detect(img_pil)

        if boxes is None or len(boxes) != 1:
            rejected += 1
            continue
        if float(probs[0]) < CONFIDENCE_THRESHOLD:
            rejected += 1
            continue

        filtered.append(fname)
        if len(filtered) == TARGET:
            break
    except Exception:
        rejected += 1

    if (i + 1) % 500 == 0:
        print(f"Checked {i+1}/{SAMPLE_SIZE} | Passed: {len(filtered)} | Rejected: {rejected}")

print(f"\nFinal set: {len(filtered)} images | Rejected: {rejected}")

Checked 500/7000 | Passed: 491 | Rejected: 9
Checked 1000/7000 | Passed: 979 | Rejected: 21
Checked 2000/7000 | Passed: 1964 | Rejected: 36
Checked 2500/7000 | Passed: 2460 | Rejected: 40
Checked 3000/7000 | Passed: 2952 | Rejected: 48
Checked 3500/7000 | Passed: 3444 | Rejected: 56
Checked 4000/7000 | Passed: 3935 | Rejected: 65
Checked 4500/7000 | Passed: 4432 | Rejected: 68
Checked 5000/7000 | Passed: 4926 | Rejected: 74

Final set: 5000 images | Rejected: 76


## 5. Generate Embeddings
BGR input, mean subtraction in raw pixel space (VGGFace2 preprocessing).

In [6]:
def embed_image_bgr(img_path):
    img = cv2.imread(img_path)
    if img is None:
        return None
    img  = cv2.resize(img, (224, 224))
    x    = torch.from_numpy(img).float().permute(2, 0, 1)
    mean = torch.tensor([91.4953, 103.8827, 131.0912]).view(3, 1, 1)
    x    = (x - mean).unsqueeze(0).to(device)
    with torch.no_grad():
        emb = model(x)[0].cpu().numpy()
    norm = np.linalg.norm(emb)
    return emb / norm if norm > 0 else emb

In [7]:
print("Generating embeddings...")
embeddings = {}
failed = 0

for i, fname in enumerate(filtered):
    emb = embed_image_bgr(f'{IMG_DIR}/{fname}')
    if emb is not None:
        embeddings[fname] = emb
    else:
        failed += 1
    if (i + 1) % 500 == 0:
        print(f"{i+1}/{len(filtered)}  failed={failed}")

print(f"Done. Embedded: {len(embeddings)} | Failed: {failed}")

Generating embeddings...
500/5000  failed=0
1000/5000  failed=0
1500/5000  failed=0
2000/5000  failed=0
2500/5000  failed=0
3000/5000  failed=0
3500/5000  failed=0
4000/5000  failed=0
4500/5000  failed=0
5000/5000  failed=0
Done. Embedded: 5000 | Failed: 0


## 6. Load Identity Mapping
Used to ensure a positive pair is always a different identity, not the same person.

In [8]:
identity_path = '/content/drive/MyDrive/CelebA/identity_CelebA.txt'
img_to_id = {}
with open(identity_path, 'r') as f:
    for line in f:
        parts = line.strip().split()
        if len(parts) == 2:
            img_to_id[parts[0]] = parts[1]
print(f"Identities loaded: {len(img_to_id)}")

Identities loaded: 202599


## 7. Generate Triplets
- Positive: highest similarity above 0.60, from a different identity
- Negative: random image with similarity below 0.30

In [9]:
THRESHOLD_POS = 0.60
THRESHOLD_NEG = 0.30

all_ids  = list(embeddings.keys())
all_embs = np.array(list(embeddings.values()))

triplets = []
skipped  = 0

for i, anchor_fname in enumerate(all_ids):
    anchor_emb = all_embs[i]
    anchor_id  = img_to_id.get(anchor_fname)

    sims = all_embs @ anchor_emb
    sims[i] = -1

    # Positive: highest similarity above threshold, different identity
    sorted_desc    = np.argsort(sims)[::-1]
    positive_fname = None
    positive_score = 0

    for idx in sorted_desc:
        if sims[idx] < THRESHOLD_POS:
            break
        candidate    = all_ids[idx]
        candidate_id = img_to_id.get(candidate)
        if anchor_id and candidate_id and anchor_id == candidate_id:
            continue
        positive_fname = candidate
        positive_score = float(sims[idx])
        break

    if positive_fname is None:
        skipped += 1
        continue

    # Negative: random image below threshold
    neg_pool = [all_ids[j] for j in np.where(sims < THRESHOLD_NEG)[0]]
    if not neg_pool:
        skipped += 1
        continue

    triplets.append({
        "anchor":         anchor_fname,
        "positive":       positive_fname,
        "positive_score": positive_score,
        "negative":       random.choice(neg_pool),
    })

print(f"Triplets generated: {len(triplets)} | Skipped: {skipped}")

save_path = '/content/drive/MyDrive/CelebA/triplets_best.json'
with open(save_path, 'w') as f:
    json.dump(triplets, f)
print(f"Saved -> {save_path}")

Triplets generated: 4509 | Skipped: 491
Saved -> /content/drive/MyDrive/CelebA/triplets_best.json


## 8. Dataset

In [10]:
class TripletDataset(Dataset):
    def __init__(self, triplets, img_dir):
        self.triplets = triplets
        self.img_dir  = img_dir

    def load_img(self, fname):
        img  = cv2.imread(f'{self.img_dir}/{fname}')
        img  = cv2.resize(img, (224, 224))
        x    = torch.from_numpy(img).float().permute(2, 0, 1)
        mean = torch.tensor([91.4953, 103.8827, 131.0912]).view(3, 1, 1)
        return x - mean

    def __len__(self):
        return len(self.triplets)

    def __getitem__(self, idx):
        t = self.triplets[idx]
        return (
            self.load_img(t['anchor']),
            self.load_img(t['positive']),
            self.load_img(t['negative']),
        )

dataset    = TripletDataset(triplets, IMG_DIR)
dataloader = DataLoader(dataset, batch_size=32, shuffle=True, num_workers=4, drop_last=True)
print(f"Dataset: {len(dataset)} triplets | Batches per epoch: {len(dataloader)}")

Dataset: 4509 triplets | Batches per epoch: 140


/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:558: UserWarning: This DataLoader will create 4 worker processes in total. Our suggested max number of worker in current system is 2, which is smaller than what this DataLoader is going to create. Please be aware that excessive worker creation might get DataLoader running slow or even freeze, lower the worker number to avoid potential slowness/freeze if necessary.
  warnings.warn(_create_warning_msg(


## 9. Fine-Tuning
Only `layer4` is trainable; all other layers remain frozen.

In [11]:
EPOCHS    = 5
LR        = 1e-5
MARGIN    = 0.3
SAVE_PATH = '/content/drive/MyDrive/CelebA/vggface2_mtcnn_filtered_final.pth'

for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Trainable parameters: {trainable:,} (layer4 only)")

optimizer = torch.optim.Adam(model.layer4.parameters(), lr=LR)
criterion = nn.TripletMarginLoss(margin=MARGIN, p=2)

model.train()
print("Starting fine-tuning...\n")

for epoch in range(EPOCHS):
    total_loss = 0
    correct    = 0
    n_batches  = 0

    for anchor, positive, negative in dataloader:
        anchor   = anchor.to(device)
        positive = positive.to(device)
        negative = negative.to(device)

        emb_a = nn.functional.normalize(model(anchor),   p=2, dim=1)
        emb_p = nn.functional.normalize(model(positive), p=2, dim=1)
        emb_n = nn.functional.normalize(model(negative), p=2, dim=1)

        loss = criterion(emb_a, emb_p, emb_n)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        d_pos = (emb_a - emb_p).pow(2).sum(1)
        d_neg = (emb_a - emb_n).pow(2).sum(1)
        correct    += (d_pos < d_neg).sum().item()
        total_loss += loss.item()
        n_batches  += 1

    avg_loss = total_loss / n_batches
    accuracy = correct / len(dataset) * 100
    print(f"Epoch {epoch+1}/{EPOCHS} | Loss: {avg_loss:.4f} | Triplet Acc: {accuracy:.1f}%")

torch.save(model.state_dict(), SAVE_PATH)
print(f"\nModel saved -> {SAVE_PATH}")

Trainable parameters: 14,964,736 (layer4 only)
Starting fine-tuning...

Epoch 1/5 | Loss: 0.0441 | Triplet Acc: 99.2%
Epoch 2/5 | Loss: 0.0054 | Triplet Acc: 99.4%
Epoch 3/5 | Loss: 0.0024 | Triplet Acc: 99.4%
Epoch 4/5 | Loss: 0.0009 | Triplet Acc: 99.4%
Epoch 5/5 | Loss: 0.0007 | Triplet Acc: 99.4%

Model saved -> /content/drive/MyDrive/CelebA/vggface2_mtcnn_filtered_final.pth


## Result

This configuration achieved **91.89% High Confidence accuracy** on the AVFS benchmark (n=185), up from an 83.78% pretrained baseline.